In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
import sys
sys.path.append("..")  # helpers.py лежит в корне курса
from helpers import add_user_message

client = Anthropic()
model = "claude-sonnet-5-5"

In [2]:
# Structured outputs вместо prefill: API возвращает JSON по схеме
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "tasks": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {"task": {"type": "string"}},
                                "required": ["task"],
                                "additionalProperties": False,
                            },
                        }
                    },
                    "required": ["tasks"],
                    "additionalProperties": False,
                },
            }
        },
    )
    text = next(block.text for block in response.content if block.type == "text")
    return json.loads(text)["tasks"]

dataset = generate_dataset()
dataset

[{'task': 'Write a Python function that takes an S3 bucket name and a prefix, and uses boto3 to return a list of all object keys under that prefix, handling pagination correctly.'},
 {'task': "Write a JSON IAM policy document that grants read-only access (s3:GetObject and s3:ListBucket) to a bucket named 'my-app-logs' and all objects within it."},
 {'task': "Write a regex that matches a valid AWS EC2 instance ID, which is 'i-' followed by either 8 or 17 lowercase hexadecimal characters, and that matches the entire string only."}]

In [3]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)